In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()


# Decision Trees

Another method different than linear regression


## Load Dataset

Let's load the clean Airbnb dataset in again 

In [2]:
file_path = f"/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)


In [3]:
from pyspark.ml.feature import StringIndexer

categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")






## VectorAssembler

Let's use the <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html?highlight=vectorassembler#pyspark.ml.feature.VectorAssembler" target="_blank">VectorAssembler</a> to combine all of our categorical and numeric inputs.


In [4]:
from pyspark.ml.feature import VectorAssembler

# Filter for just numeric columns (and exclude price, the target column)
numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price"))]

# Combine output of StringIndexer defined above and numeric columns
assembler_inputs = index_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")






## Decision Tree




In [5]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(labelCol="price")






## Training model with Pipeline

The following cell is expected to error, but we subsequently fix this.


In [6]:
from pyspark.ml import Pipeline

# Combine stages into pipeline
stages = [string_indexer, vec_assembler, dt]
pipeline = Pipeline(stages=stages)

# Train model with the train set
pipeline_model = pipeline.fit(train_df)


IllegalArgumentException: requirement failed: DecisionTree requires maxBins (= 32) to be at least as large as the number of values in each categorical feature, but categorical feature 2 has 219 values. Consider removing this and other categorical features with a large number of values, or add more training examples.

In [7]:
assembler_inputs


['host_is_superhostIndex',
 'instant_bookableIndex',
 'neighbourhood_cleansedIndex',
 'property_typeIndex',
 'room_typeIndex',
 'host_total_listings_count',
 'latitude',
 'longitude',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'minimum_nights',
 'number_of_reviews',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value',
 'bedrooms_na',
 'bathrooms_na',
 'beds_na',
 'review_scores_rating_na',
 'review_scores_accuracy_na',
 'review_scores_cleanliness_na',
 'review_scores_checkin_na',
 'review_scores_communication_na',
 'review_scores_location_na',
 'review_scores_value_na']

In [8]:
dt.setMaxBins(250)


DecisionTreeRegressor_94c91f4d4262

In [9]:
pipeline_model = pipeline.fit(train_df)






## Feature Importance




In [11]:
dt_model = pipeline_model.stages[-1]
dt_model


DecisionTreeRegressionModel: uid=DecisionTreeRegressor_94c91f4d4262, depth=5, numNodes=61, numFeatures=31

In [13]:
dt_model.featureImportances


SparseVector(31, {2: 0.1904, 3: 0.5781, 4: 0.0007, 5: 0.0613, 8: 0.0241, 10: 0.0391, 12: 0.1063})





### Interpreting Feature Importance

It's complicated to interprete features by number, let's zip it with vec_assembler to name them


In [18]:
import pandas as pd

features_df = pd.DataFrame(list(zip(vec_assembler.getInputCols(), dt_model.featureImportances)), columns=["feature", "importance"])
features_df.sort_values(by=["importance"], ascending=False)


,feature,importance
3,property_typeIndex,0.578111
2,neighbourhood_cleansedIndex,0.190432
12,minimum_nights,0.106300
5,host_total_listings_count,0.061329
10,bedrooms,0.039104
8,accommodates,0.024059
4,room_typeIndex,0.000665
0,host_is_superhostIndex,0.000000
25,review_scores_accuracy_na,0.000000
21,bedrooms_na,0.000000




# Only a handful of features are > 0

this is because default **`maxDepth`** is 5, so there are only a few features that where considered



In [19]:
top_n = 5

top_features = features_df.sort_values(["importance"], ascending=False)[:top_n]["feature"].values
print(top_features)


['property_typeIndex' 'neighbourhood_cleansedIndex' 'minimum_nights'
 'host_total_listings_count' 'bedrooms']






## Scale Invariant

With decision trees, the scale of the features does not matter. For example, it will split 1/3 of the data if that split point is 100 or if it is normalized to be .33. The only thing that matters is how many data points fall left and right of that split point - not the absolute value of the split point.

This is not true for linear regression, and the default in Spark is to standardize first. Think about it: If you measure shoe sizes in American vs European sizing, the corresponding weight of those features will be very different even those those measures represent the same thing: the size of a person's foot!






## Apply model to test set


In [20]:
pred_df = pipeline_model.transform(test_df)

pred_df.select("features", "price", "prediction").orderBy("price", ascending=False)


DataFrame[features: vector, price: double, prediction: double]



We can filter out those outliers




In [25]:
pred_df.select("features", "price", "prediction")\
.orderBy("price", ascending=False) \
.filter("prediction < 2000")\
.filter("price <2000")\
.show(truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+------------------+
|features                                                                                                                                                                 |price|prediction        |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+------------------+
|(31,[1,2,3,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],[1.0,111.0,7.0,8.0,40.7422324471234,-73.99128587488565,2.0,1.0,1.0,1.0,1.0,5.0,4.4,5.0,4.6,4.8,4.0,5.0,4.8])      |249.0|165.55343511450383|
|(31,[0,2,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],[1.0,3.0,1.0,5.0,40.76164,-73.96639,5.0,1.5,3.0,3.0,30.0,19.0,4.58,4.58,4.74,4.79,4.89,4.95,4.58])                |249.0|155.93020937188436|
|(31,[2,3,5,6,7





## Pitfall

What if we get a massive Airbnb rental? It was 20 bedrooms and 20 bathrooms. What will a decision tree predict?

It turns out decision trees cannot predict any values larger than they were trained on. The max value in our training set was $10,000, so we can't predict any values larger than that.


In [26]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse")

rmse = regression_evaluator.evaluate(pred_df)
r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")


RMSE is 41.55891855867239
R2 is 0.48208934904432266
